# Максимальное правдоподобие и MAP-оценка

## Модуль 4: Оценка параметров распределений

В этом ноутбуке мы рассмотрим два важных метода оценки параметров: MLE (Maximum Likelihood Estimation) и MAP (Maximum A Posteriori).

### Содержание:
1. Функция правдоподобия
2. Метод максимального правдоподобия (MLE)
3. Логарифмическое правдоподобие
4. MAP-оценка
5. Сравнение MLE и MAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Функция правдоподобия

**Функция правдоподобия** показывает, насколько вероятно получить наблюдаемые данные при заданных параметрах.

### Для дискретных распределений:
$$L(\theta | x_1, ..., x_n) = \prod_{i=1}^{n} P(X_i = x_i | \theta)$$

### Для непрерывных распределений:
$$L(\theta | x_1, ..., x_n) = \prod_{i=1}^{n} f(x_i | \theta)$$

**Важно:** Правдоподобие — это НЕ вероятность параметра! Это функция параметра при фиксированных данных.

**Интуитивно:** Мы ищем параметр $\theta$, при котором наблюдаемые данные наиболее вероятны.

In [ ]:
# Пример: Функция правдоподобия для монеты
# Данные: 7 орлов из 10 бросков
n_trials = 10
n_successes = 7

# Функция правдоподобия для биномиального распределения
def likelihood_binomial(p, n_successes, n_trials):
    """Правдоподобие для биномиального распределения"""
    return stats.binom.pmf(n_successes, n_trials, p)

# Диапазон значений p
p_range = np.linspace(0, 1, 1000)
likelihoods = [likelihood_binomial(p, n_successes, n_trials) for p in p_range]

# MLE-оценка
p_mle = n_successes / n_trials

print('Функция правдоподобия для монеты')
print('=' * 60)
print(f'Данные: {n_successes} орлов из {n_trials} бросков')
print(f'MLE-оценка: p̂ = {p_mle}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Правдоподобие
axes[0].plot(p_range, likelihoods, 'b-', linewidth=2)
axes[0].axvline(p_mle, color='red', linestyle='--', linewidth=2, 
                label=f'MLE: p̂ = {p_mle}')
axes[0].fill_between(p_range, likelihoods, alpha=0.3)
axes[0].set_xlabel('p')
axes[0].set_ylabel('L(p | data)')
axes[0].set_title('Функция правдоподобия')
axes[0].legend()

# Логарифмическое правдоподобие
log_likelihoods = [np.log(likelihood_binomial(p, n_successes, n_trials) + 1e-10) 
                   for p in p_range]

axes[1].plot(p_range, log_likelihoods, 'b-', linewidth=2)
axes[1].axvline(p_mle, color='red', linestyle='--', linewidth=2, 
                label=f'MLE: p̂ = {p_mle}')
axes[1].set_xlabel('p')
axes[1].set_ylabel('log L(p | data)')
axes[1].set_title('Логарифмическое правдоподобие')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Метод максимального правдоподобия (MLE)

**MLE-оценка** — это значение параметра, максимизирующее функцию правдоподобия:

$$\hat{\theta}_{MLE} = \arg\max_{\theta} L(\theta | x_1, ..., x_n)$$

или эквивалентно (для логарифмического правдоподобия):

$$\hat{\theta}_{MLE} = \arg\max_{\theta} \log L(\theta | x_1, ..., x_n)$$

### Примеры MLE-оценок:

**Нормальное распределение $N(\mu, \sigma^2)$:**
$$\hat{\mu}_{MLE} = \bar{X} = \frac{1}{n} \sum_{i=1}^{n} X_i$$
$$\hat{\sigma}^2_{MLE} = \frac{1}{n} \sum_{i=1}^{n} (X_i - \bar{X})^2$$

**Бернулли $\text{Bernoulli}(p)$:**
$$\hat{p}_{MLE} = \frac{1}{n} \sum_{i=1}^{n} X_i = \bar{X}$$

**Пуассон $\text{Poisson}(\lambda)$:**
$$\hat{\lambda}_{MLE} = \bar{X}$$

In [ ]:
# Пример: MLE для нормального распределения
np.random.seed(42)

# Генерация данных
mu_true = 5
sigma_true = 2
n_samples = 100

data = np.random.normal(mu_true, sigma_true, n_samples)

# MLE-оценки
mu_mle = np.mean(data)
sigma2_mle = np.var(data, ddof=0)  # MLE использует n, не n-1
sigma_mle = np.sqrt(sigma2_mle)

print('MLE для нормального распределения')
print('=' * 60)
print(f'Истинные параметры: μ = {mu_true}, σ = {sigma_true}')
print(f'\nMLE-оценки:')
print(f'  μ̂ = X̄ = {mu_mle:.4f}')
print(f'  σ̂² = {sigma2_mle:.4f}')
print(f'  σ̂ = {sigma_mle:.4f}')

# Визуализация правдоподобия
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Правдоподобие для μ при фиксированном σ
mu_range = np.linspace(mu_true - 3, mu_true + 3, 1000)
log_likelihoods = []
for mu in mu_range:
    ll = np.sum(stats.norm.logpdf(data, mu, sigma_mle))
    log_likelihoods.append(ll)

axes[0].plot(mu_range, log_likelihoods, 'b-', linewidth=2)
axes[0].axvline(mu_mle, color='red', linestyle='--', linewidth=2, 
                label=f'MLE: μ̂ = {mu_mle:.2f}')
axes[0].axvline(mu_true, color='green', linestyle='--', linewidth=2, 
                label=f'Истинное: μ = {mu_true}')
axes[0].set_xlabel('μ')
axes[0].set_ylabel('log L(μ | data)')
axes[0].set_title('Правдоподобие для μ (σ фиксировано)')
axes[0].legend()

# Правдоподобие для σ при фиксированном μ
sigma_range = np.linspace(0.5, 4, 1000)
log_likelihoods = []
for sigma in sigma_range:
    ll = np.sum(stats.norm.logpdf(data, mu_mle, sigma))
    log_likelihoods.append(ll)

axes[1].plot(sigma_range, log_likelihoods, 'b-', linewidth=2)
axes[1].axvline(sigma_mle, color='red', linestyle='--', linewidth=2, 
                label=f'MLE: σ̂ = {sigma_mle:.2f}')
axes[1].axvline(sigma_true, color='green', linestyle='--', linewidth=2, 
                label=f'Истинное: σ = {sigma_true}')
axes[1].set_xlabel('σ')
axes[1].set_ylabel('log L(σ | data)')
axes[1].set_title('Правдоподобие для σ (μ фиксировано)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Свойства MLE

### Асимптотические свойства:
1. **Несмещённость:** $E[\hat{\theta}_{MLE}] \to \theta$ при $n \to \infty$
2. **Состоятельность:** $\hat{\theta}_{MLE} \xrightarrow{P} \theta$ при $n \to \infty$
3. **Эффективность:** Достигает информационной границы Рао-Крамера
4. **Асимптотическая нормальность:** $\hat{\theta}_{MLE} \sim N(\theta, I(\theta)^{-1})$

### Инвариантность:
Если $\hat{\theta}$ — MLE для $\theta$, то $g(\hat{\theta})$ — MLE для $g(\theta)$.

### Проблемы:
- Может быть смещённой для малых выборок (например, $\hat{\sigma}^2_{MLE}$)
- Может не существовать (равномерное распределение)
- Может быть неединственной

In [ ]:
# Демонстрация: Смещённость MLE для дисперсии
np.random.seed(42)

# Параметры
mu_true = 0
sigma_true = 1
n_simulations = 1000
sample_sizes = [5, 10, 20, 50, 100, 500]

results = []

for n in sample_sizes:
    var_mle_list = []
    var_unbiased_list = []
    
    for _ in range(n_simulations):
        data = np.random.normal(mu_true, sigma_true, n)
        var_mle_list.append(np.var(data, ddof=0))  # MLE (n)
        var_unbiased_list.append(np.var(data, ddof=1))  # Несмещённая (n-1)
    
    results.append({
        'n': n,
        'var_mle_mean': np.mean(var_mle_list),
        'var_unbiased_mean': np.mean(var_unbiased_list),
        'var_mle_std': np.std(var_mle_list),
        'var_unbiased_std': np.std(var_unbiased_list)
    })

results_df = pd.DataFrame(results)

print('Смещённость MLE для дисперсии')
print('=' * 60)
print(f'Истинная дисперсия: σ² = {sigma_true**2}')
print()
print(results_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Средние оценки
axes[0].plot(sample_sizes, results_df['var_mle_mean'], 'ro-', linewidth=2, 
             markersize=8, label='MLE (ddof=0)')
axes[0].plot(sample_sizes, results_df['var_unbiased_mean'], 'bs-', linewidth=2, 
             markersize=8, label='Несмещённая (ddof=1)')
axes[0].axhline(sigma_true**2, color='green', linestyle='--', linewidth=2, 
                label=f'σ² = {sigma_true**2}')
axes[0].set_xlabel('Размер выборки (n)')
axes[0].set_ylabel('Средняя оценка дисперсии')
axes[0].set_title('Смещённость MLE для дисперсии')
axes[0].legend()
axes[0].set_xscale('log')

# Стандартные отклонения оценок
axes[1].plot(sample_sizes, results_df['var_mle_std'], 'ro-', linewidth=2, 
             markersize=8, label='MLE (ddof=0)')
axes[1].plot(sample_sizes, results_df['var_unbiased_std'], 'bs-', linewidth=2, 
             markersize=8, label='Несмещённая (ddof=1)')
axes[1].set_xlabel('Размер выборки (n)')
axes[1].set_ylabel('Стандартное отклонение оценки')
axes[1].set_title('Точность оценок')
axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

## 4. MAP-оценка (Maximum A Posteriori)

**MAP-оценка** учитывает априорное распределение параметра.

$$\hat{\theta}_{MAP} = \arg\max_{\theta} P(\theta | x_1, ..., x_n)$$

По теореме Байеса:

$$P(\theta | x_1, ..., x_n) = \frac{L(\theta | x_1, ..., x_n) \cdot P(\theta)}{P(x_1, ..., x_n)}$$

Так как знаменатель не зависит от $\theta$:

$$\hat{\theta}_{MAP} = \arg\max_{\theta} [L(\theta | x_1, ..., x_n) \cdot P(\theta)]$$

или в логарифмической форме:

$$\hat{\theta}_{MAP} = \arg\max_{\theta} [\log L(\theta | data) + \log P(\theta)]$$

**Связь с MLE:**
- Если $P(\theta)$ — равномерное, то MAP = MLE
- MAP = MLE + регуляризация (штраф за отклонение от априорного)

In [ ]:
# Пример: MAP для параметра монеты
np.random.seed(42)

# Данные
n_trials = 20
n_successes = 12

# MLE-оценка
p_mle = n_successes / n_trials

# Априорное распределение: Beta(2, 2)
# (слабое априорное, близкое к равномерному)
alpha_prior = 2
beta_prior = 2

# Апостериорное распределение: Beta(α + successes, β + failures)
alpha_post = alpha_prior + n_successes
beta_post = beta_prior + (n_trials - n_successes)

# MAP-оценка (мода бета-распределения)
p_map = (alpha_post - 1) / (alpha_post + beta_post - 2)

print('MAP-оценка для параметра монеты')
print('=' * 60)
print(f'Данные: {n_successes} успехов из {n_trials} испытаний')
print(f'\nMLE-оценка: p̂ = {p_mle:.4f}')
print(f'\nАприорное: Beta({alpha_prior}, {beta_prior})')
print(f'Апостериорное: Beta({alpha_post}, {beta_post})')
print(f'MAP-оценка: p̂ = {p_map:.4f}')

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

p_range = np.linspace(0, 1, 1000)

# Априорное
axes[0].plot(p_range, stats.beta.pdf(p_range, alpha_prior, beta_prior), 
             'b-', linewidth=2, label=f'Beta({alpha_prior}, {beta_prior})')
axes[0].fill_between(p_range, stats.beta.pdf(p_range, alpha_prior, beta_prior), 
                     alpha=0.3)
axes[0].set_xlabel('p')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Априорное распределение')
axes[0].legend()

# Правдоподобие
likelihoods = [stats.binom.pmf(n_successes, n_trials, p) for p in p_range]
axes[1].plot(p_range, likelihoods, 'g-', linewidth=2)
axes[1].axvline(p_mle, color='red', linestyle='--', linewidth=2, 
                label=f'MLE: {p_mle:.3f}')
axes[1].fill_between(p_range, likelihoods, alpha=0.3, color='green')
axes[1].set_xlabel('p')
axes[1].set_ylabel('L(p | data)')
axes[1].set_title('Правдоподобие')
axes[1].legend()

# Апостериорное
axes[2].plot(p_range, stats.beta.pdf(p_range, alpha_post, beta_post), 
             'r-', linewidth=2, label=f'Beta({alpha_post}, {beta_post})')
axes[2].axvline(p_map, color='blue', linestyle='--', linewidth=2, 
                label=f'MAP: {p_map:.3f}')
axes[2].axvline(p_mle, color='green', linestyle='--', linewidth=2, 
                label=f'MLE: {p_mle:.3f}')
axes[2].fill_between(p_range, stats.beta.pdf(p_range, alpha_post, beta_post), 
                     alpha=0.3, color='red')
axes[2].set_xlabel('p')
axes[2].set_ylabel('P(p | data)')
axes[2].set_title('Апостериорное распределение')
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Сравнение MLE и MAP

| Критерий | MLE | MAP |
|----------|-----|-----|
| **Априорное** | Не использует | Использует |
| **Регуляризация** | Нет | Да (через априорное) |
| **Малые выборки** | Может быть смещённой | Лучше работает |
| **Большие выборки** | Хорошие свойства | Сходится к MLE |
| **Переобучение** | Подвержена | Менее подвержена |

**Когда использовать MAP:**
- Малый объём данных
- Есть априорная информация
- Хотим избежать переобучения

**Когда использовать MLE:**
- Большой объём данных
- Нет априорной информации
- Хотим объективную оценку

In [ ]:
# Демонстрация: MLE vs MAP при разных размерах выборки
np.random.seed(42)

# Параметры
mu_true = 5
sigma_true = 1

# Априорное для μ: N(4, 1)
mu_prior = 4
sigma_prior = 1

sample_sizes = [5, 10, 20, 50, 100, 500]
n_simulations = 1000

results = []

for n in sample_sizes:
    mle_errors = []
    map_errors = []
    
    for _ in range(n_simulations):
        data = np.random.normal(mu_true, sigma_true, n)
        
        # MLE
        mu_mle = np.mean(data)
        
        # MAP (при нормальном априорном)
        # Апостериорное: N((n*x̄/σ² + μ₀/σ₀²) / (n/σ² + 1/σ₀²), 1/(n/σ² + 1/σ₀²))
        precision_data = n / sigma_true**2
        precision_prior = 1 / sigma_prior**2
        mu_map = (precision_data * np.mean(data) + precision_prior * mu_prior) / \
                 (precision_data + precision_prior)
        
        mle_errors.append((mu_mle - mu_true)**2)
        map_errors.append((mu_map - mu_true)**2)
    
    results.append({
        'n': n,
        'MSE_MLE': np.mean(mle_errors),
        'MSE_MAP': np.mean(map_errors)
    })

results_df = pd.DataFrame(results)

print('Сравнение MLE и MAP')
print('=' * 60)
print(f'Истинное μ = {mu_true}')
print(f'Априорное: N({mu_prior}, {sigma_prior}²)')
print()
print(results_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE
axes[0].plot(sample_sizes, results_df['MSE_MLE'], 'ro-', linewidth=2, 
             markersize=8, label='MLE')
axes[0].plot(sample_sizes, results_df['MSE_MAP'], 'bs-', linewidth=2, 
             markersize=8, label='MAP')
axes[0].set_xlabel('Размер выборки (n)')
axes[0].set_ylabel('MSE')
axes[0].set_title('Среднеквадратичная ошибка')
axes[0].legend()
axes[0].set_xscale('log')
axes[0].set_yscale('log')

# Отношение MSE
axes[1].plot(sample_sizes, results_df['MSE_MAP'] / results_df['MSE_MLE'], 
             'go-', linewidth=2, markersize=8)
axes[1].axhline(1, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Размер выборки (n)')
axes[1].set_ylabel('MSE_MAP / MSE_MLE')
axes[1].set_title('Отношение ошибок (MAP/MLE)')
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

print('\nВывод: MAP лучше работает при малых выборках,')
print('MLE догоняет при увеличении объёма данных.')

## Упражнения

### Упражнение 1: MLE для Пуассона
Даны данные о количестве клиентов за час:
```
[3, 5, 2, 4, 6, 3, 5, 4, 3, 5]
```

1. Найдите MLE-оценку параметра $\lambda$
2. Постройте функцию правдоподобия

### Упражнение 2: MLE для экспоненциального распределения
Даны данные о времени между заказами (в минутах):
```
[2.3, 1.5, 3.1, 0.8, 2.7, 1.9, 3.5, 2.1, 1.2, 2.8]
```

1. Найдите MLE-оценку параметра $\lambda$
2. Каково среднее время между заказами?

### Упражнение 3: MAP с разными априорными
Используя данные из упражнения 1:
1. Найдите MAP-оценку с априорным $\text{Gamma}(3, 1)$
2. Сравните с MLE
3. Как изменится MAP при априорном $\text{Gamma}(10, 2)$?

### Упражнение 4: Регуляризация через MAP
Покажите, что MAP с нормальным априорным $N(0, \lambda)$ эквивалентен Ridge-регрессии:
$$\hat{\theta}_{MAP} = \arg\min_{\theta} \left[ \sum_{i=1}^{n} (y_i - \theta x_i)^2 + \lambda \theta^2 \right]$$

---

**Решения** можно найти в ноутбуке `solutions/09_Solutions.ipynb`